# YOLO11n — Validation Error Audit

This notebook audits the **best enhanced YOLO11n detector** on the **100-image validation split only**.

It does **not** use the internal test split.

It measures:

- confidence-threshold sweep
- TP / FP / FN at IoU ≥ 0.50
- low-IoU localization misses
- matched IoU quality
- recall by relative object size
- edge / partial-object recall
- crowded-image performance
- heuristic low-light / low-contrast / blur / blue-green-cast performance
- hardest validation images
- visual examples of FN / FP / low-IoU cases
- automatic ranking of what the next YOLO experiment should focus on

> Environmental condition tags are heuristic triage signals, not manually verified labels.


In [ ]:
# CELL 1 — INSTALL
!pip -q install -U ultralytics pandas pillow matplotlib


In [ ]:
# CELL 2 — MOUNT DRIVE + IMPORTS

from google.colab import drive
drive.mount("/content/drive")

from ultralytics import YOLO
from pathlib import Path
from PIL import Image, ImageDraw

import ultralytics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import json
import cv2
from collections import defaultdict

print("=" * 100)
print("ENVIRONMENT")
print("=" * 100)
print("Ultralytics :", ultralytics.__version__)
print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab.")

DEVICE = 0
print("GPU         :", torch.cuda.get_device_name(0))


In [ ]:
# CELL 3 — PATHS

PROJECT_ROOT = Path("/content/drive/MyDrive/Coral_Reef_AI_Project")

BEST_MODEL = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Enhanced_Underwater_20Epochs"
    / "Detection_Stage2"
    / "weights"
    / "best.pt"
)

DETECTION_V1 = (
    PROJECT_ROOT
    / "01_Data"
    / "Final"
    / "Detection_v1"
)

VAL_IMAGES_DIR = DETECTION_V1 / "images" / "validation"
VAL_COCO = DETECTION_V1 / "annotations" / "instances_validation.json"

AUDIT_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Enhanced_Underwater_20Epochs"
    / "Validation_Error_Audit"
)

AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

VIS_ROOT = AUDIT_ROOT / "visuals"
for sub in ["false_negatives", "false_positives", "low_iou", "hardest_images"]:
    (VIS_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("PATH CHECK")
print("=" * 100)
print("Best model :", BEST_MODEL)
print("Val images :", VAL_IMAGES_DIR)
print("Val COCO   :", VAL_COCO)
print("Audit out  :", AUDIT_ROOT)

for p in [BEST_MODEL, VAL_IMAGES_DIR, VAL_COCO]:
    if not p.exists():
        raise FileNotFoundError(p)

val_image_paths = sorted([
    p for p in VAL_IMAGES_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
])

print("\nValidation images found:", len(val_image_paths))
assert len(val_image_paths) == 100

print("\nSTATUS: PATHS_OK")


In [ ]:
# CELL 4 — LOAD COCO GROUND TRUTH

with open(VAL_COCO, "r", encoding="utf-8") as f:
    coco = json.load(f)

images_by_id = {int(im["id"]): im for im in coco["images"]}

image_id_by_basename = {
    Path(im["file_name"]).name: int(im["id"])
    for im in coco["images"]
}

gt_by_image_id = defaultdict(list)

for ann in coco["annotations"]:
    image_id = int(ann["image_id"])
    x, y, w, h = map(float, ann["bbox"])

    gt_by_image_id[image_id].append({
        "annotation_id": int(ann["id"]),
        "bbox_xyxy": np.array([x, y, x + w, y + h], dtype=np.float32),
        "bbox_xywh": np.array([x, y, w, h], dtype=np.float32),
        "area": float(w * h),
        "category_id": int(ann["category_id"]),
    })

num_gt = sum(len(v) for v in gt_by_image_id.values())

print("COCO images :", len(coco["images"]))
print("GT boxes    :", num_gt)
print("Categories  :", coco["categories"])

assert len(coco["images"]) == 100
assert num_gt == 354

print("\nSTATUS: GT_LOADED")


In [ ]:
# CELL 5 — RUN LOW-CONFIDENCE PREDICTION ON VALIDATION

model = YOLO(str(BEST_MODEL))

PRED_MIN_CONF = 0.001
NMS_IOU = 0.70
IMGSZ = 832
MAX_DET = 300

print("=" * 100)
print("RUNNING VALIDATION PREDICTIONS")
print("=" * 100)
print("imgsz    :", IMGSZ)
print("min conf :", PRED_MIN_CONF)
print("NMS IoU  :", NMS_IOU)
print("max_det  :", MAX_DET)

results = model.predict(
    source=[str(p) for p in val_image_paths],
    imgsz=IMGSZ,
    conf=PRED_MIN_CONF,
    iou=NMS_IOU,
    max_det=MAX_DET,
    device=DEVICE,
    verbose=False,
    stream=False,
)

predictions = {}

for r in results:
    basename = Path(r.path).name

    if r.boxes is None or len(r.boxes) == 0:
        predictions[basename] = {
            "boxes": np.empty((0, 4), dtype=np.float32),
            "conf": np.empty((0,), dtype=np.float32),
        }
    else:
        predictions[basename] = {
            "boxes": r.boxes.xyxy.detach().cpu().numpy().astype(np.float32),
            "conf": r.boxes.conf.detach().cpu().numpy().astype(np.float32),
        }

print("Prediction records:", len(predictions))
assert len(predictions) == 100

print("\nSTATUS: PREDICTIONS_READY")


In [ ]:
# CELL 6 — IOU + MATCHING UTILITIES

def box_iou_matrix(boxes1, boxes2):
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    boxes1 = np.asarray(boxes1, dtype=np.float32)
    boxes2 = np.asarray(boxes2, dtype=np.float32)

    lt = np.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = np.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = np.clip(rb - lt, a_min=0, a_max=None)
    inter = wh[..., 0] * wh[..., 1]

    area1 = (
        np.clip(boxes1[:, 2] - boxes1[:, 0], 0, None)
        * np.clip(boxes1[:, 3] - boxes1[:, 1], 0, None)
    )
    area2 = (
        np.clip(boxes2[:, 2] - boxes2[:, 0], 0, None)
        * np.clip(boxes2[:, 3] - boxes2[:, 1], 0, None)
    )

    union = area1[:, None] + area2[None, :] - inter
    return inter / np.clip(union, 1e-9, None)


def greedy_match(pred_boxes, gt_boxes, iou_threshold=0.50):
    n_pred = len(pred_boxes)
    n_gt = len(gt_boxes)

    if n_pred == 0:
        return [], [], list(range(n_gt))
    if n_gt == 0:
        return [], list(range(n_pred)), []

    ious = box_iou_matrix(pred_boxes, gt_boxes)
    candidates = []

    for p in range(n_pred):
        for g in range(n_gt):
            if ious[p, g] >= iou_threshold:
                candidates.append((float(ious[p, g]), p, g))

    candidates.sort(reverse=True)

    used_p = set()
    used_g = set()
    matches = []

    for iou, p, g in candidates:
        if p in used_p or g in used_g:
            continue
        used_p.add(p)
        used_g.add(g)
        matches.append((p, g, iou))

    unmatched_preds = [p for p in range(n_pred) if p not in used_p]
    unmatched_gts = [g for g in range(n_gt) if g not in used_g]

    return matches, unmatched_preds, unmatched_gts


def pair_localization_misses(
    pred_boxes,
    gt_boxes,
    unmatched_preds,
    unmatched_gts,
    low_iou_min=0.10,
    low_iou_max=0.50,
):
    if not unmatched_preds or not unmatched_gts:
        return [], unmatched_preds.copy(), unmatched_gts.copy()

    ious = box_iou_matrix(pred_boxes, gt_boxes)
    candidates = []

    for p in unmatched_preds:
        for g in unmatched_gts:
            iou = float(ious[p, g])
            if low_iou_min <= iou < low_iou_max:
                candidates.append((iou, p, g))

    candidates.sort(reverse=True)

    used_p = set()
    used_g = set()
    pairs = []

    for iou, p, g in candidates:
        if p in used_p or g in used_g:
            continue
        used_p.add(p)
        used_g.add(g)
        pairs.append((p, g, iou))

    remaining_preds = [p for p in unmatched_preds if p not in used_p]
    remaining_gts = [g for g in unmatched_gts if g not in used_g]

    return pairs, remaining_preds, remaining_gts


def safe_precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) else 0.0

def safe_recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) else 0.0

def safe_f1(p, r):
    return 2 * p * r / (p + r) if (p + r) else 0.0


In [ ]:
# CELL 7 — CONFIDENCE THRESHOLD SWEEP

IOU_TP_THRESHOLD = 0.50
thresholds = np.round(np.arange(0.05, 0.81, 0.025), 3)

threshold_rows = []

for conf_threshold in thresholds:
    TP = FP = FN = 0

    for image_path in val_image_paths:
        basename = image_path.name
        image_id = image_id_by_basename[basename]

        gt_boxes = np.array(
            [x["bbox_xyxy"] for x in gt_by_image_id[image_id]],
            dtype=np.float32,
        ).reshape(-1, 4)

        pred = predictions[basename]
        keep = pred["conf"] >= conf_threshold
        pred_boxes = pred["boxes"][keep]

        matches, unmatched_p, unmatched_g = greedy_match(
            pred_boxes,
            gt_boxes,
            iou_threshold=IOU_TP_THRESHOLD,
        )

        TP += len(matches)
        FP += len(unmatched_p)
        FN += len(unmatched_g)

    precision = safe_precision(TP, FP)
    recall = safe_recall(TP, FN)
    f1 = safe_f1(precision, recall)

    threshold_rows.append({
        "confidence_threshold": float(conf_threshold),
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

threshold_df = pd.DataFrame(threshold_rows)
best_idx = threshold_df["f1"].idxmax()
best_threshold_row = threshold_df.loc[best_idx]
BEST_CONF = float(best_threshold_row["confidence_threshold"])

print("=" * 100)
print("CONFIDENCE THRESHOLD SWEEP")
print("=" * 100)

print(
    threshold_df.sort_values("f1", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nBest Validation-F1 confidence threshold:")
print(f"conf      = {BEST_CONF:.3f}")
print(f"Precision = {best_threshold_row['precision']:.4f}")
print(f"Recall    = {best_threshold_row['recall']:.4f}")
print(f"F1        = {best_threshold_row['f1']:.4f}")

THRESHOLD_CSV = AUDIT_ROOT / "confidence_threshold_sweep.csv"
threshold_df.to_csv(THRESHOLD_CSV, index=False)

print("\nSaved:", THRESHOLD_CSV)


In [ ]:
# CELL 8 — OBJECT + IMAGE ERROR AUDIT

def relative_size_bucket(box_xyxy, image_w, image_h):
    # Custom relative-to-frame buckets:
    # small <1%, medium 1%-<10%, large >=10% of image area.
    x1, y1, x2, y2 = map(float, box_xyxy)
    box_area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    image_area = float(image_w * image_h)
    ratio = box_area / image_area if image_area else 0.0

    if ratio < 0.01:
        bucket = "small"
    elif ratio < 0.10:
        bucket = "medium"
    else:
        bucket = "large"

    return bucket, ratio


def touches_edge(box_xyxy, image_w, image_h, margin_fraction=0.02):
    x1, y1, x2, y2 = map(float, box_xyxy)
    mx = image_w * margin_fraction
    my = image_h * margin_fraction

    return bool(
        x1 <= mx
        or y1 <= my
        or x2 >= (image_w - mx)
        or y2 >= (image_h - my)
    )


object_rows = []
prediction_rows = []
image_rows = []

for image_path in val_image_paths:
    basename = image_path.name
    image_id = image_id_by_basename[basename]

    with Image.open(image_path) as im:
        image_w, image_h = im.size

    gt_items = gt_by_image_id[image_id]
    gt_boxes = np.array(
        [x["bbox_xyxy"] for x in gt_items],
        dtype=np.float32,
    ).reshape(-1, 4)

    pred = predictions[basename]
    keep = pred["conf"] >= BEST_CONF
    pred_boxes = pred["boxes"][keep]
    pred_confs = pred["conf"][keep]

    matches, unmatched_p, unmatched_g = greedy_match(
        pred_boxes,
        gt_boxes,
        iou_threshold=IOU_TP_THRESHOLD,
    )

    localization_pairs, pure_fp, pure_fn = pair_localization_misses(
        pred_boxes,
        gt_boxes,
        unmatched_p,
        unmatched_g,
        low_iou_min=0.10,
        low_iou_max=IOU_TP_THRESHOLD,
    )

    matched_by_gt = {g: (p, iou) for p, g, iou in matches}
    localization_by_gt = {g: (p, iou) for p, g, iou in localization_pairs}
    matched_by_pred = {p: (g, iou) for p, g, iou in matches}
    localization_by_pred = {p: (g, iou) for p, g, iou in localization_pairs}

    for g, gt in enumerate(gt_items):
        size_bucket, area_ratio = relative_size_bucket(
            gt["bbox_xyxy"], image_w, image_h
        )

        edge = touches_edge(
            gt["bbox_xyxy"], image_w, image_h
        )

        if g in matched_by_gt:
            p, iou = matched_by_gt[g]
            status = "TP"
            matched_conf = float(pred_confs[p])
            matched_pred_idx = int(p)
        elif g in localization_by_gt:
            p, iou = localization_by_gt[g]
            status = "LOW_IOU"
            matched_conf = float(pred_confs[p])
            matched_pred_idx = int(p)
        else:
            iou = 0.0
            status = "FN"
            matched_conf = np.nan
            matched_pred_idx = None

        object_rows.append({
            "image": basename,
            "image_id": image_id,
            "annotation_id": gt["annotation_id"],
            "status": status,
            "gt_x1": float(gt["bbox_xyxy"][0]),
            "gt_y1": float(gt["bbox_xyxy"][1]),
            "gt_x2": float(gt["bbox_xyxy"][2]),
            "gt_y2": float(gt["bbox_xyxy"][3]),
            "relative_area": float(area_ratio),
            "size_bucket": size_bucket,
            "touches_edge": edge,
            "matched_pred_idx": matched_pred_idx,
            "matched_confidence": matched_conf,
            "iou": float(iou),
        })

    for p in range(len(pred_boxes)):
        pred_size, pred_area_ratio = relative_size_bucket(
            pred_boxes[p], image_w, image_h
        )

        if p in matched_by_pred:
            g, iou = matched_by_pred[p]
            status = "TP"
            gt_index = int(g)
        elif p in localization_by_pred:
            g, iou = localization_by_pred[p]
            status = "LOW_IOU"
            gt_index = int(g)
        else:
            iou = 0.0
            status = "FP"
            gt_index = None

        prediction_rows.append({
            "image": basename,
            "image_id": image_id,
            "prediction_index": int(p),
            "status": status,
            "confidence": float(pred_confs[p]),
            "pred_x1": float(pred_boxes[p][0]),
            "pred_y1": float(pred_boxes[p][1]),
            "pred_x2": float(pred_boxes[p][2]),
            "pred_y2": float(pred_boxes[p][3]),
            "pred_relative_area": float(pred_area_ratio),
            "pred_size_bucket": pred_size,
            "matched_gt_index": gt_index,
            "iou": float(iou),
        })

    TP = len(matches)
    low_iou = len(localization_pairs)
    FP = len(pure_fp)
    FN = len(pure_fn)

    standard_fp = FP + low_iou
    standard_fn = FN + low_iou

    p_img = safe_precision(TP, standard_fp)
    r_img = safe_recall(TP, standard_fn)
    f1_img = safe_f1(p_img, r_img)

    matched_ious = [x[2] for x in matches]

    image_rows.append({
        "image": basename,
        "image_id": image_id,
        "width": image_w,
        "height": image_h,
        "gt_count": len(gt_boxes),
        "pred_count": len(pred_boxes),
        "TP": TP,
        "LOW_IOU": low_iou,
        "PURE_FP": FP,
        "PURE_FN": FN,
        "standard_FP": standard_fp,
        "standard_FN": standard_fn,
        "precision": p_img,
        "recall": r_img,
        "f1": f1_img,
        "mean_matched_iou": float(np.mean(matched_ious)) if matched_ious else np.nan,
    })

object_df = pd.DataFrame(object_rows)
prediction_df = pd.DataFrame(prediction_rows)
image_df = pd.DataFrame(image_rows)

OBJECT_CSV = AUDIT_ROOT / "object_audit.csv"
PRED_CSV = AUDIT_ROOT / "prediction_audit.csv"
IMAGE_CSV = AUDIT_ROOT / "per_image_audit.csv"

object_df.to_csv(OBJECT_CSV, index=False)
prediction_df.to_csv(PRED_CSV, index=False)
image_df.to_csv(IMAGE_CSV, index=False)

print("=" * 100)
print("AUDIT COUNTS")
print("=" * 100)
print("Audit confidence threshold :", BEST_CONF)
print("\nGT status:")
print(object_df["status"].value_counts().to_string())

print("\nPrediction status:")
print(prediction_df["status"].value_counts().to_string())


In [ ]:
# CELL 9 — SIZE / EDGE / CROWDING ANALYSIS

def gt_group_metrics(group):
    gt_count = len(group)
    tp = int((group["status"] == "TP").sum())
    low_iou = int((group["status"] == "LOW_IOU").sum())
    pure_fn = int((group["status"] == "FN").sum())

    recall_iou50 = tp / gt_count if gt_count else 0.0
    localization_failure_rate = low_iou / gt_count if gt_count else 0.0
    pure_miss_rate = pure_fn / gt_count if gt_count else 0.0

    matched = group[group["status"] == "TP"]

    return pd.Series({
        "gt_count": gt_count,
        "TP": tp,
        "LOW_IOU": low_iou,
        "PURE_FN": pure_fn,
        "recall_iou50": recall_iou50,
        "localization_failure_rate": localization_failure_rate,
        "pure_miss_rate": pure_miss_rate,
        "mean_tp_iou": matched["iou"].mean() if len(matched) else np.nan,
    })

size_metrics = (
    object_df
    .groupby("size_bucket", observed=True)
    .apply(gt_group_metrics)
    .reset_index()
)

edge_metrics = (
    object_df
    .groupby("touches_edge", observed=True)
    .apply(gt_group_metrics)
    .reset_index()
)

image_df["crowded"] = image_df["gt_count"] >= 4

crowded_metrics = (
    image_df
    .groupby("crowded", observed=True)
    .agg(
        images=("image", "count"),
        gt_count=("gt_count", "sum"),
        TP=("TP", "sum"),
        FP=("standard_FP", "sum"),
        FN=("standard_FN", "sum"),
    )
    .reset_index()
)

crowded_metrics["precision"] = (
    crowded_metrics["TP"]
    / (crowded_metrics["TP"] + crowded_metrics["FP"]).clip(lower=1)
)

crowded_metrics["recall"] = (
    crowded_metrics["TP"]
    / (crowded_metrics["TP"] + crowded_metrics["FN"]).clip(lower=1)
)

crowded_metrics["f1"] = (
    2 * crowded_metrics["precision"] * crowded_metrics["recall"]
    / (crowded_metrics["precision"] + crowded_metrics["recall"]).replace(0, np.nan)
)

SIZE_CSV = AUDIT_ROOT / "performance_by_relative_object_size.csv"
EDGE_CSV = AUDIT_ROOT / "performance_edge_vs_nonedge.csv"
CROWD_CSV = AUDIT_ROOT / "performance_crowded_vs_noncrowded.csv"

size_metrics.to_csv(SIZE_CSV, index=False)
edge_metrics.to_csv(EDGE_CSV, index=False)
crowded_metrics.to_csv(CROWD_CSV, index=False)

print("=" * 100)
print("RELATIVE OBJECT SIZE")
print("=" * 100)
print(size_metrics.to_string(index=False))

print("\n" + "=" * 100)
print("EDGE / PARTIAL OBJECTS")
print("=" * 100)
print(edge_metrics.to_string(index=False))

print("\n" + "=" * 100)
print("CROWDED IMAGES")
print("=" * 100)
print(crowded_metrics.to_string(index=False))


In [ ]:
# CELL 10 — IMAGE CONDITION HEURISTICS
# These are triage signals, not manually verified labels.

condition_rows = []

for image_path in val_image_paths:
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if img is None:
        raise RuntimeError(f"Could not read {image_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    mean_brightness = float(gray.mean())
    contrast_std = float(gray.std())
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    mean_saturation = float(hsv[..., 1].mean())

    b_mean = float(img[..., 0].mean())
    g_mean = float(img[..., 1].mean())
    r_mean = float(img[..., 2].mean())

    bg_to_r = float(
        (b_mean + g_mean)
        / (2.0 * max(r_mean, 1e-6))
    )

    condition_rows.append({
        "image": image_path.name,
        "mean_brightness": mean_brightness,
        "contrast_std": contrast_std,
        "laplacian_variance": lap_var,
        "mean_saturation": mean_saturation,
        "blue_green_to_red_ratio": bg_to_r,
    })

condition_df = pd.DataFrame(condition_rows)

q25_brightness = condition_df["mean_brightness"].quantile(0.25)
q25_contrast = condition_df["contrast_std"].quantile(0.25)
q25_sharpness = condition_df["laplacian_variance"].quantile(0.25)
q75_cast = condition_df["blue_green_to_red_ratio"].quantile(0.75)

condition_df["low_light_q25"] = condition_df["mean_brightness"] <= q25_brightness
condition_df["low_contrast_q25"] = condition_df["contrast_std"] <= q25_contrast
condition_df["low_sharpness_q25"] = condition_df["laplacian_variance"] <= q25_sharpness
condition_df["strong_blue_green_cast_q75"] = (
    condition_df["blue_green_to_red_ratio"] >= q75_cast
)

condition_merged = image_df.merge(condition_df, on="image", how="left")

condition_names = [
    "low_light_q25",
    "low_contrast_q25",
    "low_sharpness_q25",
    "strong_blue_green_cast_q75",
]

condition_summary_rows = []

for condition in condition_names:
    for flag in [True, False]:
        g = condition_merged[condition_merged[condition] == flag]

        TP = int(g["TP"].sum())
        FP = int(g["standard_FP"].sum())
        FN = int(g["standard_FN"].sum())

        p = safe_precision(TP, FP)
        r = safe_recall(TP, FN)
        f1 = safe_f1(p, r)

        condition_summary_rows.append({
            "condition": condition,
            "flag": flag,
            "images": len(g),
            "GT": int(g["gt_count"].sum()),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

condition_summary_df = pd.DataFrame(condition_summary_rows)

CONDITION_RAW_CSV = AUDIT_ROOT / "image_condition_heuristics.csv"
CONDITION_SUMMARY_CSV = AUDIT_ROOT / "performance_by_condition_heuristic.csv"

condition_merged.to_csv(CONDITION_RAW_CSV, index=False)
condition_summary_df.to_csv(CONDITION_SUMMARY_CSV, index=False)

print("=" * 100)
print("IMAGE CONDITION HEURISTICS")
print("=" * 100)
print("Bottom quartile brightness :", round(q25_brightness, 3))
print("Bottom quartile contrast   :", round(q25_contrast, 3))
print("Bottom quartile sharpness  :", round(q25_sharpness, 3))
print("Top quartile B/G-to-red    :", round(q75_cast, 3))

print("\nPerformance:")
print(condition_summary_df.to_string(index=False))

print(
    "\nNOTE: These condition labels are automatic heuristics, "
    "not manually verified environmental annotations."
)


In [ ]:
# CELL 11 — LOCALIZATION QUALITY

tp_objects = object_df[object_df["status"] == "TP"].copy()

localization_summary = {
    "TP_count": int(len(tp_objects)),
    "mean_TP_IoU": float(tp_objects["iou"].mean()) if len(tp_objects) else None,
    "median_TP_IoU": float(tp_objects["iou"].median()) if len(tp_objects) else None,
    "TP_IoU_0.50_to_0.75": int(
        ((tp_objects["iou"] >= 0.50) & (tp_objects["iou"] < 0.75)).sum()
    ),
    "TP_IoU_ge_0.75": int((tp_objects["iou"] >= 0.75).sum()),
    "LOW_IOU_pairs_0.10_to_0.50": int(
        (object_df["status"] == "LOW_IOU").sum()
    ),
}

print("=" * 100)
print("LOCALIZATION QUALITY")
print("=" * 100)

for k, v in localization_summary.items():
    print(f"{k:<30}: {v}")


In [ ]:
# CELL 12 — RANK HARDEST VALIDATION IMAGES

image_df["hardness_score"] = (
    image_df["PURE_FN"] * 3.0
    + image_df["LOW_IOU"] * 2.5
    + image_df["PURE_FP"] * 1.5
    + (1.0 - image_df["mean_matched_iou"].fillna(0.0))
)

hardest_df = image_df.sort_values(
    ["hardness_score", "standard_FN", "standard_FP"],
    ascending=False,
).reset_index(drop=True)

HARDEST_CSV = AUDIT_ROOT / "hardest_validation_images.csv"
hardest_df.to_csv(HARDEST_CSV, index=False)

print("=" * 100)
print("TOP 20 HARDEST VALIDATION IMAGES")
print("=" * 100)

cols = [
    "image",
    "gt_count",
    "pred_count",
    "TP",
    "LOW_IOU",
    "PURE_FN",
    "PURE_FP",
    "mean_matched_iou",
    "hardness_score",
]

print(hardest_df[cols].head(20).to_string(index=False))
print("\nSaved:", HARDEST_CSV)


In [ ]:
# CELL 13 — SAVE ERROR VISUALIZATIONS

def draw_box(draw, box, color, width=4):
    x1, y1, x2, y2 = map(float, box)
    draw.rectangle([x1, y1, x2, y2], outline=color, width=width)


def annotate_image(image_name, output_path):
    image_path = VAL_IMAGES_DIR / image_name
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    gt_objects = object_df[object_df["image"] == image_name]
    pred_objects = prediction_df[prediction_df["image"] == image_name]

    for _, row in gt_objects.iterrows():
        gt_box = [row["gt_x1"], row["gt_y1"], row["gt_x2"], row["gt_y2"]]
        draw_box(draw, gt_box, color="lime", width=4)
        draw.text(
            (gt_box[0] + 4, gt_box[1] + 4),
            f"GT {row['status']} | {row['size_bucket']}",
            fill="lime",
        )

    for _, row in pred_objects.iterrows():
        pred_box = [
            row["pred_x1"],
            row["pred_y1"],
            row["pred_x2"],
            row["pred_y2"],
        ]

        if row["status"] == "LOW_IOU":
            color = "orange"
        elif row["status"] == "TP":
            color = "deepskyblue"
        else:
            color = "red"

        draw_box(draw, pred_box, color=color, width=3)
        draw.text(
            (pred_box[0] + 4, pred_box[1] + 20),
            f"P {row['status']} c={row['confidence']:.2f} IoU={row['iou']:.2f}",
            fill=color,
        )

    image.save(output_path, quality=92)


for _, row in hardest_df.head(20).iterrows():
    annotate_image(
        row["image"],
        VIS_ROOT / "hardest_images" / row["image"],
    )

fn_images = (
    object_df.loc[object_df["status"] == "FN", "image"]
    .value_counts().head(15).index.tolist()
)

fp_images = (
    prediction_df.loc[prediction_df["status"] == "FP", "image"]
    .value_counts().head(15).index.tolist()
)

low_iou_images = (
    object_df.loc[object_df["status"] == "LOW_IOU", "image"]
    .value_counts().head(15).index.tolist()
)

for image_name in fn_images:
    annotate_image(image_name, VIS_ROOT / "false_negatives" / image_name)

for image_name in fp_images:
    annotate_image(image_name, VIS_ROOT / "false_positives" / image_name)

for image_name in low_iou_images:
    annotate_image(image_name, VIS_ROOT / "low_iou" / image_name)

print("Visual audit folders:")
print(VIS_ROOT)

print("\nLegend:")
print("Green  = Ground Truth")
print("Blue   = TP prediction")
print("Orange = Low-IoU prediction")
print("Red    = Pure false positive")


In [ ]:
# CELL 14 — PLOTS

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    threshold_df["confidence_threshold"],
    threshold_df["precision"],
    label="Precision",
)
ax.plot(
    threshold_df["confidence_threshold"],
    threshold_df["recall"],
    label="Recall",
)
ax.plot(
    threshold_df["confidence_threshold"],
    threshold_df["f1"],
    label="F1",
)
ax.axvline(
    BEST_CONF,
    linestyle="--",
    label=f"Best F1 conf={BEST_CONF:.3f}",
)

ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Score")
ax.set_title("YOLO11n Validation — Confidence Threshold Sweep")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()

THRESHOLD_PLOT = AUDIT_ROOT / "confidence_threshold_sweep.png"
fig.savefig(THRESHOLD_PLOT, dpi=180, bbox_inches="tight")
plt.show()


size_order = ["small", "medium", "large"]

size_plot_df = (
    size_metrics
    .set_index("size_bucket")
    .reindex(size_order)
    .dropna(how="all")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    size_plot_df["size_bucket"],
    size_plot_df["recall_iou50"],
)
ax.set_ylim(0, 1)
ax.set_xlabel("Relative object size")
ax.set_ylabel("Recall @ IoU 0.50")
ax.set_title("YOLO11n Validation — Recall by Object Size")
ax.grid(True, axis="y", alpha=0.3)

fig.tight_layout()

SIZE_PLOT = AUDIT_ROOT / "recall_by_relative_object_size.png"
fig.savefig(SIZE_PLOT, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:")
print(THRESHOLD_PLOT)
print(SIZE_PLOT)


In [ ]:
# CELL 15 — AUTOMATIC FOCUS RECOMMENDATIONS

focus_rows = []

total_gt = len(object_df)

low_iou_rate = (
    (object_df["status"] == "LOW_IOU").sum()
    / total_gt
)

tp_under_075_rate = (
    (
        (object_df["status"] == "TP")
        & (object_df["iou"] < 0.75)
    ).sum()
    / total_gt
)

localization_score = low_iou_rate + tp_under_075_rate

focus_rows.append({
    "focus": "Bounding-box localization",
    "difficulty_score": float(localization_score),
    "evidence": (
        f"LOW_IOU rate={low_iou_rate:.3f}, "
        f"TP with IoU<0.75 per GT={tp_under_075_rate:.3f}"
    ),
    "next_training_direction": (
        "Lower LR fine-tuning; reduce/disable Mosaic-MixUp-CutMix; "
        "use fixed high resolution; verify GT box tightness."
    ),
})

for _, row in size_metrics.iterrows():
    focus_rows.append({
        "focus": f"{row['size_bucket'].title()} coral",
        "difficulty_score": float(1.0 - row["recall_iou50"]),
        "evidence": (
            f"GT={int(row['gt_count'])}, "
            f"Recall={row['recall_iou50']:.3f}, "
            f"LOW_IOU rate={row['localization_failure_rate']:.3f}"
        ),
        "next_training_direction": (
            "Prioritize this size group only if it is materially weaker "
            "than the other size groups."
        ),
    })

for _, row in edge_metrics.iterrows():
    label = "Edge/partial coral" if bool(row["touches_edge"]) else "Non-edge coral"

    focus_rows.append({
        "focus": label,
        "difficulty_score": float(1.0 - row["recall_iou50"]),
        "evidence": (
            f"GT={int(row['gt_count'])}, "
            f"Recall={row['recall_iou50']:.3f}"
        ),
        "next_training_direction": (
            "If edge objects are weak, preserve partial objects and use "
            "moderate crop/translation augmentation."
        ),
    })

condition_label_map = {
    "low_light_q25": "Low-light images",
    "low_contrast_q25": "Low-contrast / visibility images",
    "low_sharpness_q25": "Blur / low-detail images",
    "strong_blue_green_cast_q75": "Strong blue/green cast",
}

for condition in condition_names:
    positive = condition_summary_df[
        (condition_summary_df["condition"] == condition)
        & (condition_summary_df["flag"] == True)
    ]

    negative = condition_summary_df[
        (condition_summary_df["condition"] == condition)
        & (condition_summary_df["flag"] == False)
    ]

    if len(positive) and len(negative):
        pos = positive.iloc[0]
        neg = negative.iloc[0]
        recall_gap = float(neg["recall"] - pos["recall"])

        focus_rows.append({
            "focus": condition_label_map[condition],
            "difficulty_score": float(max(0.0, recall_gap)),
            "evidence": (
                f"Recall condition={pos['recall']:.3f}, "
                f"Recall others={neg['recall']:.3f}, "
                f"gap={recall_gap:+.3f}"
            ),
            "next_training_direction": (
                "Strengthen this underwater augmentation only if the "
                "measured recall gap is clearly negative."
            ),
        })

focus_df = pd.DataFrame(focus_rows).sort_values(
    "difficulty_score",
    ascending=False,
).reset_index(drop=True)

FOCUS_CSV = AUDIT_ROOT / "recommended_focus_ranking.csv"
focus_df.to_csv(FOCUS_CSV, index=False)

print("=" * 110)
print("NEXT YOLO FOCUS — RANKED FROM MEASURED VALIDATION FAILURES")
print("=" * 110)

print(
    focus_df[
        ["focus", "difficulty_score", "evidence"]
    ].head(10).to_string(index=False)
)

print("\nSaved:", FOCUS_CSV)
print(
    "\nIMPORTANT: Do not increase every augmentation blindly. "
    "Use the measured weak groups above."
)


In [ ]:
# CELL 16 — FINAL AUDIT SUMMARY

best_sweep = threshold_df.loc[threshold_df["f1"].idxmax()]

summary = {
    "model": str(BEST_MODEL),
    "validation_images": int(len(val_image_paths)),
    "validation_gt_boxes": int(len(object_df)),
    "prediction_imgsz": int(IMGSZ),
    "match_iou_threshold": float(IOU_TP_THRESHOLD),
    "best_validation_f1_confidence_threshold": float(BEST_CONF),
    "precision_at_best_f1": float(best_sweep["precision"]),
    "recall_at_best_f1": float(best_sweep["recall"]),
    "f1_at_best_f1": float(best_sweep["f1"]),
    "TP": int((object_df["status"] == "TP").sum()),
    "LOW_IOU": int((object_df["status"] == "LOW_IOU").sum()),
    "PURE_FN": int((object_df["status"] == "FN").sum()),
    "PURE_FP": int((prediction_df["status"] == "FP").sum()),
    "mean_TP_IoU": float(tp_objects["iou"].mean()) if len(tp_objects) else None,
    "median_TP_IoU": float(tp_objects["iou"].median()) if len(tp_objects) else None,
    "test_evaluated": False,
    "top_focus": focus_df.iloc[0]["focus"] if len(focus_df) else None,
}

SUMMARY_JSON = AUDIT_ROOT / "audit_summary.json"

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("=" * 110)
print("YOLO11n VALIDATION ERROR AUDIT — FINAL SUMMARY")
print("=" * 110)

for k, v in summary.items():
    print(f"{k:<42}: {v}")

print("\nOutputs:")
for p in [
    THRESHOLD_CSV,
    OBJECT_CSV,
    PRED_CSV,
    IMAGE_CSV,
    SIZE_CSV,
    EDGE_CSV,
    CROWD_CSV,
    CONDITION_RAW_CSV,
    CONDITION_SUMMARY_CSV,
    HARDEST_CSV,
    FOCUS_CSV,
    SUMMARY_JSON,
    VIS_ROOT,
]:
    print(p)

print("\nINTERNAL TEST SET WAS NOT USED.")
print("\nSTATUS: YOLO11N_VALIDATION_ERROR_AUDIT_COMPLETE")
